In [ ]:
from apiclient import discovery
from httplib2 import Http
from oauth2client import client, file, tools
import csv

created_forms = []

In [ ]:
SCOPES = [
    "https://www.googleapis.com/auth/forms.body",
    "https://www.googleapis.com/auth/forms.responses.readonly"
]

DISCOVERY_DOC = "https://forms.googleapis.com/$discovery/rest?version=v1"

store = file.Storage("token.json")
creds = None

if not creds or creds.invalid:
  flow = client.flow_from_clientsecrets("client_secrets.json", SCOPES)
  flags = tools.argparser.parse_args([])   # or parse_known_args([])[0]
  creds = tools.run_flow(flow, store, flags)
  
form_service = discovery.build(
    "forms",
    "v1",
    http=creds.authorize(Http()),
    discoveryServiceUrl=DISCOVERY_DOC,
    static_discovery=False,
)



In [ ]:
# Reload registration CSV
SIGNUPS_CSV = "2026 JAGM Registration! Due February 24th (Responses) - Form Responses 1.csv"
EMAIL_Q = 'Your *WATERLOO* Email'
SOC_Q = 'Your Society'

email_to_soc = {}

with open(SIGNUPS_CSV, mode='r') as file:
    # Use DictReader to read the file
    reader = csv.DictReader(file)

    # Convert rows into a list of dictionaries
    for row in reader:
        email_to_soc[row[EMAIL_Q]] = row[SOC_Q][0]

print(email_to_soc)


In [ ]:
# Create Motion Voting Form
title = input("Title of motion form:")
description = input("Description:")
motion = input("Motion")

if (len(title) == 0):
    raise Exception("NO TITLE SET")

# Request body for creating a form
NEW_FORM = {
    "info": {
        "title": title,
        "documentTitle": title,
    }
}

# Request body to add a multiple-choice question
NEW_QUESTION = {
    "requests": [
        {
            "updateFormInfo": {
                "info": {
                    "description": description,
                },
                "updateMask": "description"
            }
        },
        {
            "createItem": {
                "item": {
                    "itemId": "00000000",
                    "title": (
                        "Waterloo Email (e.g. j2gordon@uwaterloo.ca)"
                    ),
                    "questionItem": {
                        "question": {
                            "questionId" : "69",
                            "required": True,
                            "textQuestion": {
                                "paragraph": False
                            },
                        }
                    },
                },
                "location": {"index": 0},
            }
        },
        {
            "createItem": {
                "item": {
                    "itemId": "00000001",
                    "title": (
                        "How do you vote on the motion: " + motion
                    ),
                    "questionItem": {
                        "question": {
                            "questionId":"67",
                            "required": True,
                            "choiceQuestion": {
                                "type": "RADIO",
                                "options": [
                                    {"value": "In Favour"},
                                    {"value": "Opposed"},
                                    {"value": "Abstain"},
                                ],
                                "shuffle": True,
                            },
                        }
                    },
                },
                "location": {"index": 1},
            }
        }
    ]
}

if title != "":
    # Creates the initial form
    result = form_service.forms().create(body=NEW_FORM).execute()

    # sets the questions
    question_setting = (
        form_service.forms()
        .batchUpdate(formId=result["formId"], body=NEW_QUESTION)
        .execute()
    )

    # gets final data
    get_result = form_service.forms().get(formId=result["formId"]).execute()
    print(get_result)

    created_forms.insert(0, [get_result["info"]["title"], get_result["formId"]])

In [ ]:
# Create Election Voting Form
title = input("Title of election form:")
description = input("Description:")
motion = input("Motion")
candidates = []
while "" not in candidates:
    candidates.append(input("Enter a candidate name. Press enter to stop. No Candidate will be added automatically."))
candidates[-1] = "No Candidate"

if (len(title) == 0):
    raise Exception("NO TITLE SET")

# Request body for creating a form
NEW_FORM = {
    "info": {
        "title": title,
        "documentTitle": title,
    }
}

# Request body to add a multiple-choice question
NEW_QUESTION = {
    "requests": [
        {
            "updateFormInfo": {
                "info": {
                    "description": description,
                },
                "updateMask": "description"
            }
        },
        {
            "createItem": {
                "item": {
                    "itemId": "00000000",
                    "title": (
                        "Waterloo Email (e.g. j2gordon@uwaterloo.ca)"
                    ),
                    "questionItem": {
                        "question": {
                            "questionId" : "69",
                            "required": True,
                            "textQuestion": {
                                "paragraph": False
                            },
                        }
                    },
                },
                "location": {"index": 0},
            }
        },
        {
            "createItem" : {
                "item" : {
                    "itemId": "00000001",
                    "title" : "Rank the following candidates for motion: " + motion,
                    "description": "Your favourite candidate should be ranked 1. Do not rank the same candidate multiple times -- it will not be counted.",
                    "questionGroupItem" : {
                        "questions": [
                            {
                                "questionId" : str(x+420),
                                "rowQuestion": {
                                    "title": str(x)
                                    
                                }
                            }
                            for x in range(1, len(candidates) + 1)
                        ],
                        "grid" : {
                            "columns" : {
                                "type" : "RADIO", 
                                "options" : [
                                    {
                                        "value": x
                                    }
                                    for x in candidates                                    
                                ]
                            }
                        }
                    }
                },
                "location": {
                    "index": 1
                }
            }
        }
    ]
}

if title != "":
    # Creates the initial form
    result = form_service.forms().create(body=NEW_FORM).execute()

    # sets the questions
    question_setting = (
        form_service.forms()
        .batchUpdate(formId=result["formId"], body=NEW_QUESTION)
        .execute()
    )

    # gets final data
    get_result = form_service.forms().get(formId=result["formId"]).execute()
    print(get_result)

    created_forms.insert(0, [get_result["info"]["title"], get_result["formId"]])

In [ ]:
# get results from a form
form_nums = "0 - enter other id"
for num, form in enumerate(created_forms):
    form_nums += "\n" + str(num+1) + " - " + form[0]

print(form_nums)
print("--------------------------------------")
selected = input("Enter the form number you wish to use.")

id = "0"
if selected == "0":
    id = input("Please enter new form id")
else:
    id = created_forms[int(selected) - 1][1]

responses = form_service.forms().responses().list(formId=id).execute()
print("FORM FOUND!")
answers = [x["answers"] for x in responses['responses']]
c_soc_answers = []
a_soc_answers = {}
b_soc_answers = {}
for x in answers:
    if x['00000069']['textAnswers']['answers'][0]['value'] not in email_to_soc.keys():
        c_soc_answers.append(x['00000069']['textAnswers']['answers'][0]['value'])
    elif not x['00000069']['textAnswers']['answers'][0]['value'] in c_soc_answers and email_to_soc[x['00000069']['textAnswers']['answers'][0]['value']] == "A":
        a_soc_answers[x['00000069']['textAnswers']['answers'][0]['value']] = x
    elif not x['00000069']['textAnswers']['answers'][0]['value'] in c_soc_answers and email_to_soc[x['00000069']['textAnswers']['answers'][0]['value']] == "B":
        b_soc_answers[x['00000069']['textAnswers']['answers'][0]['value']] = x
    else:
        print("SOMETHING BROKE", x)
        
print(len(a_soc_answers.keys()), "ASoc Votes")
print(len(b_soc_answers.keys()), "BSoc Votes")
print("INVALID ANSWERS:", len(c_soc_answers))
for x in (c_soc_answers):
    print(x)
print(a_soc_answers)
print(b_soc_answers)


In [ ]:
# Motion results
a_soc_results = {}
total_a_soc = 0
for a in a_soc_answers.keys():
    if a_soc_answers[a]['00000067']['textAnswers']['answers'][0]['value'] not in a_soc_results.keys():
        a_soc_results[a_soc_answers[a]['00000067']['textAnswers']['answers'][0]['value']] = 1
    else:
        a_soc_results[a_soc_answers[a]['00000067']['textAnswers']['answers'][0]['value']] += 1
    total_a_soc += 1
print("A-SOC:", a_soc_results, "out of a total of", total_a_soc)
print("FINAL A-SOC:", [(x, str(100*a_soc_results[x]/total_a_soc)+'%') for x in a_soc_results.keys()])
b_soc_results = {}
total_b_soc = 0
for b in b_soc_answers.keys():
    if b_soc_answers[b]['00000067']['textAnswers']['answers'][0]['value'] not in b_soc_results.keys():
        b_soc_results[b_soc_answers[b]['00000067']['textAnswers']['answers'][0]['value']] = 1
    else:
        b_soc_results[b_soc_answers[b]['00000067']['textAnswers']['answers'][0]['value']] += 1
    total_b_soc += 1
print("b-SOC: ", b_soc_results, "out of a total of", total_b_soc)
print("FINAL A-SOC:", [(x, str(100*b_soc_results[x]/total_b_soc)+'%') for x in b_soc_results.keys()])

In [ ]:
# Election Results

# TODO: FINISH
# should have seperate a-soc b-soc and different one where they're accumulitive
print("Percent Anu: 0%")
print("Percent Pilot: 69%")
print("Percent No Candidate: 100%")